# Chapter 11: Quantum Cohomology

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 11, printed pp. 417-486; PDF pp. 432-501. Sections: 11.1-11.5.

    ## Chapter Goal

    Small quantum cohomology, Gromov-Witten potential, examples, Seidel representation, and Frobenius manifolds.

    The guiding question is:

    > How do curve counts deform the ordinary cup product into a new associative product?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| quantum product | multiplication table with q-corrections | associativity |
| ordinary cup product | degree-respecting classical term | q=0 limit |
| GW potential | generating function of invariants | third derivative structure |
| Seidel element | invertible quantum class from a loop | unit and inverse check |
| Frobenius structure | metric plus associative product | WDVV-style constraint |


## Standalone Reading Guide

    Quantum cohomology changes the product, not the underlying vector space. The deformation records counts of curves meeting cycles, so the product remembers symplectic geometry that ordinary cohomology cannot see. In the smallest example, CP1 has a degree-two class H whose classical square vanishes, but the quantum product gives H*H=q. The symbol q records the sphere class that supplies the correction.

Associativity is the central structural fact. Geometrically it comes from comparing different degenerations of four-pointed curves, which is why gluing from the previous chapter is essential. Algebraically it appears as a multiplication law satisfying the same parenthesizing identities as ordinary cup product. The Gromov-Witten potential packages the structure constants into a generating function, and Frobenius manifolds organize the resulting family of products.

The notebook builds the CP1 quantum ring at q=1 as a two-dimensional algebra. That specialization is small enough for direct matrix checks. The associativity assertion becomes a finite computation over the basis {1, H}, while the displayed table reminds the learner where the curve correction enters.

    ## Topics In This Notebook

    - small quantum product as cup product plus curve corrections
- Gromov-Witten potential as a generating function
- examples including projective spaces and products
- Seidel representation from Hamiltonian loops
- Frobenius manifolds and associativity equations

    ## Visualization Storyboard

    - A CP1 multiplication table shows the relation H*H=q and its classical limit.
- A dependency graph links Gromov-Witten counts, gluing, and associativity.
- A ledger checks unit, grading, and associativity for a finite quantum ring model.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-11'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['quantum product', 'ordinary cup product', 'GW potential', 'Seidel element', 'Frobenius structure']
CONCEPT_EDGES = [('quantum product', 'ordinary cup product'), ('ordinary cup product', 'GW potential'), ('GW potential', 'Seidel element'), ('Seidel element', 'Frobenius structure')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=28, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Quantum Cohomology')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '417-486',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Quantum Cohomology. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
q = 1
basis = ["1", "H"]

def multiply(a, b):
    if a == "1":
        return {"1": 1 if b == "1" else 0, "H": 1 if b == "H" else 0}
    if b == "1":
        return {"1": 0, "H": 1}
    return {"1": q, "H": 0}

def vec(label):
    return np.array([1, 0]) if label == "1" else np.array([0, 1])

def mul_vec(x, y):
    out = np.zeros(2, dtype=int)
    for i, ai in enumerate(basis):
        for j, bj in enumerate(basis):
            coeff = x[i] * y[j]
            prod = multiply(ai, bj)
            out += coeff * np.array([prod["1"], prod["H"]])
    return out

table = np.zeros((2, 2), dtype=int)
for i, ai in enumerate(basis):
    for j, bj in enumerate(basis):
        prod = multiply(ai, bj)
        table[i, j] = prod["1"] + 2 * prod["H"]

fig, ax = plt.subplots(figsize=(5.6, 4.8))
im = ax.imshow(table, cmap="PuBuGn")
ax.set_xticks(range(2), basis)
ax.set_yticks(range(2), basis)
for i in range(2):
    for j in range(2):
        prod = multiply(basis[i], basis[j])
        label = "1" if prod["1"] else ("H" if prod["H"] else "0")
        ax.text(j, i, label, ha="center", va="center", fontsize=13)
ax.set_title("CP1 quantum product at q=1")
fig.colorbar(im, ax=ax, shrink=0.75)
fig_path = save_matplotlib(fig, UNIT, "figures", "cp1-quantum-product-table.png")
plt.close(fig)

assoc_ok = True
for a in basis:
    for b in basis:
        for c in basis:
            left = mul_vec(mul_vec(vec(a), vec(b)), vec(c))
            right = mul_vec(vec(a), mul_vec(vec(b), vec(c)))
            assoc_ok = assoc_ok and np.array_equal(left, right)

check = {
    "basis": basis,
    "q": q,
    "H_times_H": "q * 1",
    "associative": bool(assoc_ok),
    "unit_works": bool(np.array_equal(mul_vec(vec("1"), vec("H")), vec("H"))),
    "passed": bool(assoc_ok),
}
check_path = save_json(check, UNIT, "checks", "quantum-product-checks.json")
display_artifact(fig_path, width=540)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'quantum product', 'computational_object': 'multiplication table with q-corrections', 'check': 'associativity'}, {'item': 'ordinary cup product', 'computational_object': 'degree-respecting classical term', 'check': 'q=0 limit'}, {'item': 'GW potential', 'computational_object': 'generating function of invariants', 'check': 'third derivative structure'}, {'item': 'Seidel element', 'computational_object': 'invertible quantum class from a loop', 'check': 'unit and inverse check'}, {'item': 'Frobenius structure', 'computational_object': 'metric plus associative product', 'check': 'WDVV-style constraint'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Set q to zero to recover the classical ring, then rerun the associativity check. Associativity remains, but the deformation term disappears.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Quantum cohomology deforms cup product using Gromov-Witten counts.
- Gluing explains associativity by comparing boundary degenerations.
- Small examples such as CP1 make the q-correction completely visible.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'cp1-quantum-product-table.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'quantum-product-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'quantum-product-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
